# StormEngine — Seasonal Analysis of Mean Sea Level Pressure (MSL) — Full Year 2024

Seasonal analysis of MSL across the entire 2024 calendar year, built directly from the 12 monthly NetCDF files (one per month, each a multi-band raster of shape `(744 or fewer, 31, 33)` with variables `msl`, `u10`, `v10`, `i10fg`).

## File structure
Each NetCDF covers one calendar month at 0.25° resolution over the domain `[39–46.5°N, 12–20°E]`:
- `msl`, `u10`, `v10`, `i10fg` — hourly, dims `(valid_time, latitude, longitude)`
- All pressure values converted from **Pa → hPa** (÷100)

All four seasons (Winter, Spring, Summer, Autumn) are now covered, unlike the earlier 4-month assessment limited to Nov–Feb.

## 0. Imports and Configuration

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 11, 'font.family': 'sans-serif',
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.7,
    'legend.frameon': False,
})

# ── Input files: 12 monthly NetCDF ──
PROJECT_ROOT = r'D:\StormEngine'
DATA_DIR     = os.path.join(PROJECT_ROOT, 'dati_storici', 'estratto_std')

VARIABLE  = 'MSL'
UNIT      = 'hPa'
PA_TO_HPA = 1.0 / 100.0

LAT_MIN, LAT_MAX = 39.0, 46.5
LON_MIN, LON_MAX = 12.0, 20.0

MONTH_NAMES = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
SEASON_MAP = {12:'Winter', 1:'Winter', 2:'Winter',
              3:'Spring', 4:'Spring', 5:'Spring',
              6:'Summer', 7:'Summer', 8:'Summer',
              9:'Autumn', 10:'Autumn', 11:'Autumn'}

# Distinct colour per month (12 months, perceptually spread)
MONTH_COLORS = {m: c for m, c in zip(MONTH_NAMES, plt.cm.tab20(np.linspace(0, 1, 12)))}
SEASON_COLORS = {'Winter': '#08519c', 'Spring': '#41ab5d',
                  'Summer': '#cb181d', 'Autumn': '#fb8d3d'}

print('Configuration loaded.')
print(f'Data directory: {DATA_DIR}')

Configuration loaded.
Data directory: D:\StormEngine\dati_storici\estratto_std


## 1. Load and Verify All 12 Monthly NetCDF Files

In [2]:
nc_files = [
    os.path.join(DATA_DIR, f'era5_std_adriatico_2024_{month:02d}.nc')
    for month in range(1, 13)
]

print(f'Looking in: {DATA_DIR}')
print(f'Expected {len(nc_files)} monthly files:')
missing = []
for f in nc_files:
    exists = os.path.isfile(f)
    print(f'  {"OK" if exists else "MISSING":8s} {f}')
    if not exists:
        missing.append(f)

assert not missing, f'Missing {len(missing)} file(s): {missing}'

datasets = [xr.open_dataset(f) for f in nc_files]
ds_year = xr.concat(datasets, dim='valid_time')
ds_year = ds_year.sortby('valid_time')

print(f'\nConcatenated dataset:')
print(f'  Total hours : {ds_year.sizes["valid_time"]}  (expected 8784 for 2024, leap year)')
print(f'  Grid        : {ds_year.sizes["latitude"]} x {ds_year.sizes["longitude"]}')
print(f'  Time range  : {ds_year.valid_time.values[0]} -> {ds_year.valid_time.values[-1]}')
print(f'  Variables   : {list(ds_year.data_vars)}')

Looking in: D:\StormEngine\dati_storici\estratto_std
Expected 12 monthly files:
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_01.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_02.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_03.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_04.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_05.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_06.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_07.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_08.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_09.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_10.nc
  OK       D:\StormEngine\dati_storici\estratto_std\era5_std_adriatico_2024_11.nc
  OK       D:\Stor

## 2. Extract MSL, Convert Units, and Split by Month

Extracts the `msl` variable, converts Pa → hPa, and reorganises the continuous yearly array into one entry per calendar month — mirroring the `data` dict structure used by the rest of this notebook, but now derived directly from the NetCDF rather than from QGIS-exported CSVs.

In [3]:
# Convert MSL to hPa
msl_year = ds_year['msl'].values * PA_TO_HPA          # (n_hours, n_lat, n_lon)
valid_times = pd.DatetimeIndex(ds_year['valid_time'].values)
lats = ds_year['latitude'].values
lons = ds_year['longitude'].values

print(f'MSL array shape : {msl_year.shape}  (hours, lat, lon)')
print(f'Domain mean MSL : {np.nanmean(msl_year):.2f} {UNIT}')

# Flatten the spatial grid (lat, lon) -> single 'points' axis, matching the
# (n_points, n_hours) shape used by load_month() in the original notebook
n_hours = msl_year.shape[0]
n_points = msl_year.shape[1] * msl_year.shape[2]
samples_flat = msl_year.reshape(n_hours, n_points).T   # (n_points, n_hours)

lon_grid, lat_grid = np.meshgrid(lons, lats)
lon_flat = lon_grid.flatten()
lat_flat = lat_grid.flatten()

# Split into one dict entry per month, matching the original `data` structure
data = {}
months_idx = valid_times.month
for m in range(1, 13):
    mask = months_idx == m
    month_name = MONTH_NAMES[m-1]
    data[month_name] = {
        'value'   : np.nanmean(samples_flat[:, mask], axis=1),   # per-point mean as reference
        'samples' : samples_flat[:, mask],                        # (n_points, n_hours_in_month)
        'lon'     : lon_flat,
        'lat'     : lat_flat,
        'n_points': n_points,
        'n_hours' : int(mask.sum()),
        'season'  : SEASON_MAP[m],
    }
    print(f'{month_name:10s} | season={SEASON_MAP[m]:7s} | '
          f'points={n_points} | hours={mask.sum()} | '
          f'domain mean={data[month_name]["samples"].mean():.2f} {UNIT}')

KeyError: "No variable named 'msl'. Variables on the dataset include ['ssrd', 'number', 'valid_time', 'latitude', 'longitude', 'expver']"

## 3. Summary Statistics per Month

Domain-averaged statistics: each hourly sample is first averaged over all grid points, giving one time series per month, then summarised.

In [ ]:
rows = []
for month, d in data.items():
    ts = np.nanmean(d['samples'], axis=0)
    rows.append({
        'Month'  : month,
        'Season' : d['season'],
        'Mean'   : np.mean(ts),
        'Std'    : np.std(ts),
        'Min'    : np.min(ts),
        'Max'    : np.max(ts),
        'Range'  : np.max(ts) - np.min(ts),
        'Median' : np.median(ts),
    })

summary = pd.DataFrame(rows).set_index('Month').round(2)
print(f'Domain-averaged {VARIABLE} statistics ({UNIT})\n')
print(summary.to_string())

## 4. Monthly Time Series — Domain-Averaged MSL

Hourly evolution of the domain-mean pressure for each month, revealing the passage of synoptic systems (high/low pressure cycles), now across all 12 months.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 14), sharey=True)
axes = axes.flatten()

for ax, (month, d) in zip(axes, data.items()):
    ts   = np.nanmean(d['samples'], axis=0)
    hours = np.arange(len(ts))
    days  = hours / 24.0

    ax.plot(days, ts, color=MONTH_COLORS[month], linewidth=1.1)
    ax.axhline(ts.mean(), color='gray', linestyle='--', linewidth=0.9,
               label=f'mean = {ts.mean():.1f}')
    ax.fill_between(days, ts, ts.mean(), where=(ts >= ts.mean()),
                    alpha=0.15, color=MONTH_COLORS[month])
    ax.fill_between(days, ts, ts.mean(), where=(ts < ts.mean()),
                    alpha=0.15, color='red')

    ax.set_title(f'{month} ({d["season"]})', fontsize=11)
    ax.set_xlabel('Day of month', fontsize=9)
    ax.legend(loc='upper right', fontsize=7)

for i in range(0, len(axes), 3):
    axes[i].set_ylabel(f'{VARIABLE} ({UNIT})')

fig.suptitle(f'Domain-Averaged {VARIABLE} — Hourly Time Series, 2024',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('msl_monthly_timeseries_2024.png', bbox_inches='tight')
plt.show()

## 5. Aggregated Continuous Time Series — Full Year

All 12 months placed on a single continuous hourly timeline, in chronological order.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

cursor = 0
boundaries = []
tick_pos = []
tick_lab = []

for month in MONTH_NAMES:
    d = data[month]
    ts = np.nanmean(d['samples'], axis=0)
    days = cursor + np.arange(len(ts)) / 24.0

    ax.plot(days, ts, color=MONTH_COLORS[month], linewidth=1.0)
    ax.hlines(ts.mean(), days[0], days[-1],
              color=MONTH_COLORS[month], linestyle='--', linewidth=1.0, alpha=0.8)

    tick_pos.append(cursor + len(ts)/24.0 / 2)
    tick_lab.append(month[:3])
    cursor += len(ts) / 24.0
    boundaries.append(cursor)

for b in boundaries[:-1]:
    ax.axvline(b, color='gray', linestyle=':', linewidth=0.7, alpha=0.6)

global_mean = np.concatenate([np.nanmean(d['samples'], axis=0) for d in data.values()]).mean()
ax.axhline(global_mean, color='black', linewidth=1, alpha=0.5,
           label=f'annual mean = {global_mean:.1f} {UNIT}')

ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_lab)
ax.set_xlim(0, cursor)
ax.set_ylabel(f'{VARIABLE} ({UNIT})')
ax.set_xlabel('Month (continuous hourly timeline, 2024)')
ax.set_title(f'Aggregated {VARIABLE} Across the Full Year 2024')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('msl_aggregated_timeline_2024.png', bbox_inches='tight')
plt.show()

## 6. Distribution Comparison — Violin + Box Plots

Distribution of all hourly domain-mean values per month, across the full year.

In [ ]:
months   = MONTH_NAMES
ts_all   = [np.nanmean(data[m]['samples'], axis=0) for m in months]
colors   = [MONTH_COLORS[m] for m in months]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

parts = ax1.violinplot(ts_all, showmeans=True, showmedians=False)
for pc, c in zip(parts['bodies'], colors):
    pc.set_facecolor(c); pc.set_alpha(0.65); pc.set_edgecolor('black')
for key in ['cmeans', 'cmaxes', 'cmins', 'cbars']:
    if key in parts:
        parts[key].set_edgecolor('black'); parts[key].set_linewidth(1.0)
ax1.set_xticks(range(1, len(months)+1))
ax1.set_xticklabels([m[:3] for m in months], fontsize=9)
ax1.set_ylabel(f'{VARIABLE} ({UNIT})')
ax1.set_title('Distribution (Violin) — 2024')

bp = ax2.boxplot(ts_all, patch_artist=True, widths=0.6,
                 medianprops=dict(color='black', linewidth=1.5))
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.65)
ax2.set_xticks(range(1, len(months)+1))
ax2.set_xticklabels([m[:3] for m in months], fontsize=9)
ax2.set_ylabel(f'{VARIABLE} ({UNIT})')
ax2.set_title('Distribution (Box) — 2024')

fig.suptitle(f'{VARIABLE} Distribution Comparison Across the Full Year',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('msl_distributions_2024.png', bbox_inches='tight')
plt.show()

## 7. Diurnal Cycle — Average MSL by Hour of Day

The semidiurnal atmospheric tide produces a characteristic twice-daily oscillation in surface pressure. Now shown across all four seasons rather than only autumn/winter.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for month in months:
    d = data[month]
    ts = np.nanmean(d['samples'], axis=0)
    n_full_days = len(ts) // 24
    ts_days = ts[:n_full_days * 24].reshape(n_full_days, 24)

    diurnal_mean = ts_days.mean(axis=0)
    anomaly = diurnal_mean - diurnal_mean.mean()

    hours = np.arange(24)
    ax.plot(hours, anomaly, marker='o', markersize=3,
            color=MONTH_COLORS[month], linewidth=1.3, alpha=0.85,
            label=f'{month[:3]} ({d["season"]})')

ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel('Hour of day (UTC)')
ax.set_ylabel(f'{VARIABLE} anomaly ({UNIT})')
ax.set_title('Diurnal Cycle — MSL Anomaly Relative to Daily Mean, 2024')
ax.legend(loc='upper right', ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig('msl_diurnal_cycle_2024.png', bbox_inches='tight')
plt.show()

## 8. Spatial Maps — Seasonal Mean MSL Field

The time-averaged pressure field reconstructed on the spatial grid, one map per season (rather than per month, to keep the figure readable across the full year).

In [ ]:
def to_grid(values, lon, lat):
    valid = ~(np.isnan(lon) | np.isnan(lat))
    values, lon, lat = values[valid], lon[valid], lat[valid]
    lons_u = np.sort(np.unique(lon))
    lats_u = np.sort(np.unique(lat))[::-1]
    grid = np.full((len(lats_u), len(lons_u)), np.nan)
    lon_idx = {v: i for i, v in enumerate(lons_u)}
    lat_idx = {v: i for i, v in enumerate(lats_u)}
    for v, lo, la in zip(values, lon, lat):
        grid[lat_idx[la], lon_idx[lo]] = v
    return grid, lons_u, lats_u

# Pool months by season for the spatial mean
season_means = {}
for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    months_in_season = [m for m in months if data[m]['season'] == season]
    pooled_samples = np.concatenate([data[m]['samples'] for m in months_in_season], axis=1)
    season_means[season] = np.nanmean(pooled_samples, axis=1)

all_means = list(season_means.values())
vmin = min(np.nanmin(m) for m in all_means)
vmax = max(np.nanmax(m) for m in all_means)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

for ax, season in zip(axes, ['Winter', 'Spring', 'Summer', 'Autumn']):
    grid, lons_u, lats_u = to_grid(season_means[season], lon_flat, lat_flat)

    im = ax.imshow(grid, cmap='RdYlBu_r', vmin=vmin, vmax=vmax,
                   extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
                   aspect='auto', origin='upper')
    cs = ax.contour(grid, levels=8, colors='black', linewidths=0.5, alpha=0.4,
                    extent=[LON_MIN, LON_MAX, LAT_MAX, LAT_MIN])
    ax.clabel(cs, inline=True, fontsize=7, fmt='%.0f')
    ax.set_title(f'{season} 2024')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    plt.colorbar(im, ax=ax, label=f'{VARIABLE} ({UNIT})', shrink=0.85)

fig.suptitle(f'Seasonal Mean {VARIABLE} Spatial Field, 2024', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('msl_spatial_maps_2024.png', bbox_inches='tight')
plt.show()

## 9. Spatial Variability — Standard Deviation Maps

Where does pressure vary most over time within each season? High-variability regions indicate areas swept by moving fronts and synoptic systems.

In [ ]:
season_stds = {}
for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    months_in_season = [m for m in months if data[m]['season'] == season]
    pooled_samples = np.concatenate([data[m]['samples'] for m in months_in_season], axis=1)
    season_stds[season] = np.nanstd(pooled_samples, axis=1)

smax = max(np.nanmax(s) for s in season_stds.values())

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

for ax, season in zip(axes, ['Winter', 'Spring', 'Summer', 'Autumn']):
    grid, lons_u, lats_u = to_grid(season_stds[season], lon_flat, lat_flat)

    im = ax.imshow(grid, cmap='viridis', vmin=0, vmax=smax,
                   extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
                   aspect='auto', origin='upper')
    ax.set_title(f'{season} 2024')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    plt.colorbar(im, ax=ax, label=f'Std ({UNIT})', shrink=0.85)

fig.suptitle(f'Temporal Variability of {VARIABLE} (Std per Grid Point), 2024',
             fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('msl_variability_maps_2024.png', bbox_inches='tight')
plt.show()

## 10. Seasonal Aggregation — All Four Seasons

Pooling months by season to compare the overall pressure regime across Winter, Spring, Summer, and Autumn — now with full annual coverage.

In [ ]:
season_ts = {}
for month in months:
    s = data[month]['season']
    ts = np.nanmean(data[month]['samples'], axis=0)
    season_ts.setdefault(s, []).append(ts)

season_pooled = {s: np.concatenate(v) for s, v in season_ts.items()}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1.4, 1]})

season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
for s in season_order:
    pooled = season_pooled[s]
    ax1.hist(pooled, bins=50, alpha=0.55, density=True,
             color=SEASON_COLORS[s],
             label=f'{s}  (μ={pooled.mean():.1f}, σ={pooled.std():.1f})')
ax1.set_xlabel(f'{VARIABLE} ({UNIT})')
ax1.set_ylabel('Density')
ax1.set_title('Seasonal MSL Distribution — 2024')
ax1.legend()

s_means = [season_pooled[s].mean() for s in season_order]
s_stds  = [season_pooled[s].std()  for s in season_order]
bars = ax2.bar(season_order, s_means, yerr=s_stds, capsize=6,
               color=[SEASON_COLORS[s] for s in season_order],
               alpha=0.75, edgecolor='black', linewidth=0.8)
ax2.set_ylabel(f'Mean {VARIABLE} ({UNIT})')
ax2.set_title('Seasonal Mean ± Std — 2024')
ax2.set_ylim(min(s_means) - max(s_stds) - 2, max(s_means) + max(s_stds) + 2)
for bar, m in zip(bars, s_means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{m:.1f}', ha='center', fontsize=10, fontweight='bold')

fig.suptitle(f'{VARIABLE} — Seasonal Comparison, Full Year 2024',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('msl_seasonal_comparison_2024.png', bbox_inches='tight')
plt.show()

## 11. Key Findings

Auto-generated summary of the full-year seasonal analysis.

In [ ]:
print('='*60)
print(f'  {VARIABLE} SEASONAL ANALYSIS — KEY FINDINGS (2024)')
print('='*60)

var_by_month = {m: np.nanmean(data[m]['samples'], axis=0).std() for m in months}
most_var  = max(var_by_month, key=var_by_month.get)
least_var = min(var_by_month, key=var_by_month.get)

mean_by_month = {m: np.nanmean(data[m]['samples']) for m in months}
high_m = max(mean_by_month, key=mean_by_month.get)
low_m  = min(mean_by_month, key=mean_by_month.get)

print(f'\n  Temporal variability (synoptic activity):')
print(f'    Most variable : {most_var}  (σ={var_by_month[most_var]:.2f} {UNIT})')
print(f'    Least variable: {least_var}  (σ={var_by_month[least_var]:.2f} {UNIT})')

print(f'\n  Mean pressure level:')
print(f'    Highest: {high_m}  ({mean_by_month[high_m]:.2f} {UNIT})')
print(f'    Lowest : {low_m}  ({mean_by_month[low_m]:.2f} {UNIT})')

print(f'\n  Seasonal means (full year 2024):')
for s in ['Winter', 'Spring', 'Summer', 'Autumn']:
    pooled = season_pooled[s]
    print(f'    {s:7s}: {pooled.mean():.2f} ± {pooled.std():.2f} {UNIT}')

print(f'\n  Generated figures:')
for f in ['msl_monthly_timeseries_2024', 'msl_aggregated_timeline_2024',
          'msl_distributions_2024', 'msl_diurnal_cycle_2024',
          'msl_spatial_maps_2024', 'msl_variability_maps_2024',
          'msl_seasonal_comparison_2024']:
    print(f'    - {f}.png')
print('='*60)